# Упражнения к семинару 10. Предобработка данных: выбросы и пропуски

Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-10-tasks.ipynb`, например `2MP9-Ivanov-Ivan-seminar-10-tasks.ipynb`: этот файл сдаётся в Moodle. Формулировки упражнений здесь те же, что в листке `seminar-10-tasks.md`; дополнительно в ноутбуке заготовки функций, тесты и ячейки «Проверка».

Данные: таблицы Московской области за 2022–2025 годы, ДТП `datasets/legal/dtp-moscow-region.csv` и участники `datasets/legal/dtp-moscow-region-participants.csv`, в двух проверках таблица ДТП Липецкой области; путь в ячейках задан от папки `exercises/`, поэтому ноутбук должен лежать в ней, а папка `datasets` рядом, как в репозитории.

- Часть А: выполните ячейку с подключением библиотек, загрузите таблицу ДТП в упражнении А1 и таблицу участников в А2, затем решайте упражнения по порядку и вводите числа в тест Moodle «Семинар 10».
- Части Б и В: напишите тело функции вместо `pass`, запустите ячейку с тестами; если всё верно, она напечатает `Все тесты прошли успешно!`. Затем выполните ячейку «Проверка»: она вызывает функцию на таблице области и печатает числа, которые вводятся в тест вместе с кодом функции.

Правила: функция не ссылается на переменные, определённые вне неё, кроме `pd`, `np`, `plt`, `sns` и функций части Б; ячейки с тестами не меняются; в коде решений библиотеки, кроме pandas, NumPy, matplotlib и seaborn, не используются.

## Часть А. Разминка

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')       # оформление графиков по умолчанию
pd.set_option('display.width', 200)   # широкие таблицы печатаются без переноса

### А1. Крайние значения

Загрузите таблицу ДТП `datasets/legal/dtp-moscow-region.csv` в переменную `df` с `index_col='id'` и `parse_dates=['datetime']`. Посмотрите сводку четырёх счётчиков `df[['participants_count', 'vehicles_count', 'dead_count', 'injured_count']].describe()` и три ДТП с наибольшим числом погибших `df.nlargest(3, 'dead_count')`. К каждому крайнему значению задайте три вопроса раздела 1 ноутбука занятия: возможно ли оно, повторяется ли, объяснимо ли; найдите и ДТП с наибольшим числом участников, посмотрите его вид и район. Какое наибольшее число участников в одном ДТП и какое наибольшее число раненых? Сколько участников и сколько машин в ДТП с наибольшим числом погибших? Подсказка: раздел 1 ноутбука занятия; максимумы — строка `max` сводки, участники и машины — столбцы `participants_count` и `vehicles_count` первой строки `nlargest`. `sort_values('dead_count')` без `ascending=False` ставит первыми ДТП без погибших; максимум машин стоит в другом столбце сводки, чем максимум участников; номер ДТП из `idxmax` в этом упражнении не нужен.

In [ ]:
# А1

### А2. Границы IQR

Загрузите таблицу участников `datasets/legal/dtp-moscow-region-participants.csv` в `people` и присоедините к каждому участнику дату его ДТП: `people = people.merge(df[['datetime']], left_on='dtp_id', right_index=True)`. Возраст — свойство машины, а в таблице участников машина повторяется в строке каждого, кто в ней ехал; поэтому, как в разделе 1 ноутбука занятия, возьмите таблицу «одна строка — одна машина»: `vehicles = people.dropna(subset=['vehicle_no']).drop_duplicates(subset=['dtp_id', 'vehicle_no'])`. Добавьте в `vehicles` столбец `vehicle_age` — год ДТП минус `vehicle_year` — и возьмите возраст машин с известным годом выпуска: `age = vehicles['vehicle_age'].dropna()`, должно получиться 25 379 значений. Посчитайте квартили `q1, q3 = age.quantile([0.25, 0.75])`, размах `iqr = q3 - q1` и границы `q1 - 1.5 * iqr` и `q3 + 1.5 * iqr`; рядом нарисуйте ящик `sns.boxplot(x=age, ax=ax)` на осях `fig, ax = plt.subplots(figsize=(8, 3))`: ус кончается на последнем значении внутри границы, поэтому левый ус стоит на самой молодой машине, а не на нижней границе. Какова верхняя граница, сколько машин старше неё и какова нижняя граница? Подсказка: раздел 2 ноутбука занятия; машин старше границы — сумма маски `age > upper`. Нижняя граница ниже нуля: для возраста она бессмысленна, правило работает только сверху. Квартили можно взять и из `age.describe()`, числа те же; `iqr` — разность квартилей, а не третий квартиль; множитель 1,5, а не 3. По строкам `people`, без таблицы машин, старая машина посчитается столько раз, сколько людей в ней ехало, и машин старше границы выйдет больше. Если считать по столбцу `vehicles['vehicle_age']` с пропусками, число машин старше границы то же, но маски `> upper` и `<= upper` вместе дадут меньше строк, чем в таблице: сравнение с пропуском всегда `False`.

In [ ]:
# А2

### А3. Правило трёх сигм и процентиль

На том же возрасте машин `age` посчитайте z-оценку `z = (age - age.mean()) / age.std()` и число машин с `z.abs() > 3`, затем 99-й процентиль `age.quantile(0.99)` и число машин старше него. Сравните три числа: правило IQR из А2, правило трёх сигм и процентиль. Сколько машин отбирает правило трёх сигм? Каков возраст 99-го процентиля? Подсказка: раздел 2 ноутбука занятия; `abs()` берёт модуль: у возраста машины левого хвоста нет, но правило по модулю общее для любых столбцов. `quantile(99)` вместо `0.99` даёт `ValueError`: процентиль задаётся долей. Один процент от числа машин — не то же самое, что число машин старше 99-го процентиля: процентиль — граница, и значения ровно на ней выбросами не считаются.

In [ ]:
# А3

### А4. Редкое, но настоящее

Самые старые машины: `vehicles.nsmallest(3, 'vehicle_year')[['brand', 'vehicle_category', 'vehicle_year']]`. Сколько машин старше 50 лет по `vehicle_age`? Каков год выпуска самой старой машины? Затем стаж: замените коды 95, 96 и 99 пропуском в новом столбце `people['experience_clean']`, как в разделе 5 ноутбука занятия, и посмотрите, где кончаются настоящие значения и начинаются коды: `people['experience'].value_counts().sort_index().tail(8)`. Сколько водителей со стажем больше 60 лет? Подсказка: раздел 1 ноутбука занятия; машин старше 50 лет — сумма маски `vehicles['vehicle_age'] > 50`: по таблице машин, а не по строкам участников; редкое значение не ошибка, если оно возможно и объяснимо. Без замены кодов «стаж больше 60» получат и все коды; `nsmallest` по `vehicle_age` вместо `vehicle_year` даёт самые новые машины; границу между настоящим значением и кодом показывает словарь данных, правило выбросов её не найдёт.

In [ ]:
# А4

### А5. Логические проверки

Три логические проверки таблицы ДТП: пострадавших, то есть погибших и раненых вместе, больше участников; машин больше участников; машин нет. Каждая проверка — маска, число нарушений — её сумма. У второй маски посмотрите, какие это ДТП: `df.loc[маска, 'category'].value_counts()`. Сколько ДТП, где машин больше участников? Сколько из них наездов на стоящее ТС? Сколько ДТП, где пострадавших больше участников? Подсказка: раздел 6 ноутбука занятия; пострадавшие — `df['dead_count'] + df['injured_count']`, участники — столбец `participants_count`, а не число строк таблицы участников. `>=` вместо `>` засчитает в нарушения ДТП, где пострадавших ровно столько, сколько участников, а это возможно; маски, соединённые без скобок, дают ошибку; проверка, которая ничего не нашла, — тоже ответ.

In [ ]:
# А5

### А6. Повторы

Повторы в таблице участников: `people.duplicated(keep=False).sum()` — сколько строк входят в группы одинаковых; `people.duplicated().sum()` — сколько строк снял бы `drop_duplicates`. Маска `dup = people.duplicated()` отбирает именно снимаемые строки: посмотрите их роли и здоровье через `people.loc[dup, 'role'].value_counts()` и `people.loc[dup, 'health'].value_counts()`. Таблицу `people` не меняйте: результат `drop_duplicates` в этом упражнении никуда не присваивается. Сколько строк удалит `drop_duplicates`? Сколько из них пассажиров? Сколько раненых? Подсказка: раздел 6 ноутбука занятия; повтор не ошибка, если он объясним: пассажиры одной машины одного пола с одинаковым здоровьем неотличимы. `keep=False` помечает все строки каждой группы, а не только снимаемые; роли по всей таблице, а не по маске, отвечают на другой вопрос; если `drop_duplicates` присвоен обратно в `people`, следующее упражнение посчитается по неполной таблице.

In [ ]:
# А6

### А7. Пропуски со смыслом

Доля пропуска стажа по ролям: `pd.crosstab(people['role'], people['experience_clean'].isna(), normalize='index')`. Возьмите водителей `drivers = people[people['role'] == 'Водитель']` и посчитайте среднее `experience_clean` как есть, после заполнения пропусков медианой стажа водителей и после заполнения нулём. Какова доля пропуска стажа у водителей, до тысячных? Каково среднее стажа водителей после заполнения медианой и после заполнения нулём, до сотых? Подсказка: раздел 5 ноутбука занятия; доля у водителей — ячейка строки «Водитель» и столбца `True`; заполнение — `fillna` у столбца водителей, например `drivers['experience_clean'].fillna(0).mean()`. `normalize='columns'` даёт доли внутри пропуска, а не внутри роли; `fillna` по всей таблице участников выдаёт стаж пешеходам; среднее по `experience` с кодами — не стаж; `inplace=True` у столбца в pandas 3 таблицу не меняет.

In [ ]:
# А7

## Часть Б. Основные задачи

### Б1. Границы IQR

Правило IQR знакомо по ящику с усами семинара 8: от первого квартиля вниз и от третьего вверх откладывается полтора размаха, а ус доходит до последнего значения внутри этих границ. Одна функция вместо четырёх строк для каждого столбца отчёта, и её границы нужны функциям Б2 и В1. Пропуски в расчёт не входят: `quantile` их пропускает.

Реализуйте функцию `iqr_bounds(values, factor=1.5)`, которая считает границы выбросов по правилу межквартильного размаха. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `factor` — множитель размаха, число; по умолчанию 1.5. Возвращаемое значение: кортеж из двух чисел (lower, upper): q1 - factor * iqr и q3 + factor * iqr, где q1 и q3 — первый и третий квартили values, iqr = q3 - q1; пропуски не учитываются. Series не меняется.

Подсказка: раздел 2 ноутбука занятия: `q1, q3 = values.quantile([0.25, 0.75])` раскладывает два квартиля в две переменные, `return lower, upper` возвращает кортеж. `np.percentile` на Series с пропуском даёт `nan`.

Проверка в ноутбуке: стаж водителей области без кодов через iqr_bounds; печатаются нижняя и верхняя границы, обе вводятся в тест.

In [ ]:
def iqr_bounds(values, factor=1.5):
    """
    Считает границы выбросов по правилу межквартильного размаха.

    Правило IQR знакомо по ящику с усами семинара 8: от первого квартиля вниз и
    от третьего вверх откладывается полтора размаха, а ус доходит до последнего
    значения внутри этих границ. Одна функция вместо четырёх строк для каждого
    столбца отчёта, и её границы нужны функциям Б2 и В1. Пропуски в расчёт не
    входят: `quantile` их пропускает.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        factor: Множитель размаха, число; по умолчанию 1.5.

    Возвращаемое значение:
        Кортеж из двух чисел (lower, upper): q1 - factor * iqr и q3 + factor *
        iqr, где q1 и q3 — первый и третий квартили values, iqr = q3 - q1;
        пропуски не учитываются. Series не меняется.
    """

    pass

In [ ]:
def iqr_bounds_tests():
    values = pd.Series([1, 2, 3, 4, 100, None])
    before = values.copy()
    result = iqr_bounds(values)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж из двух чисел'
    assert result == (-1.0, 7.0), 'границы q1 - 1.5 * iqr и q3 + 1.5 * iqr, пропуск не учитывается'
    assert iqr_bounds(values, factor=3) == (-4.0, 10.0), 'множитель размаха factor'
    assert iqr_bounds(pd.Series([10, 20, 30, 40, 50])) == (-10.0, 70.0)
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
iqr_bounds_tests()

In [ ]:
# Проверка: стаж водителей области без кодов через iqr_bounds; печатаются нижняя и верхняя границы, обе вводятся в тест

people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель']

lower, upper = iqr_bounds(drivers_check['experience_clean'])
print(lower, upper)

### Б2. Маска выбросов по правилу

Три правила в одной функции, чтобы сравнивать их на одном столбце: в разделе 2 ноутбука занятия они дали три разных числа. Пропуск не выброс: сравнение с пропуском и так даёт `False`, а маска через `~` и `between` объявила бы его выбросом. Правило трёх сигм на малой выборке слепо: среди 1, 2, 3, 4 и 100 у сотни z-оценка всего 1,79.

Реализуйте функцию `flag_outliers(values, method='iqr')`, которая отмечает выбросы по правилу IQR, трёх сигм или процентилей. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `method` — правило, строка: 'iqr' — за границами iqr_bounds с множителем 1.5; 'z' — z-оценка (значение минус среднее, делённое на стандартное отклонение) по модулю больше 3; 'percentile' — ниже 1-го или выше 99-го процентиля. По умолчанию 'iqr'. Возвращаемое значение: объект Series из логических значений с тем же индексом: True у выбросов, False у остальных значений и у пропусков; значение ровно на границе — не выброс. Для другого method — ValueError. Series не меняется.

Подсказка: разделы 2 и 3 ноутбука занятия: границы IQR даёт `iqr_bounds` из Б1; маска с двух сторон — `(values < lower) | (values > upper)`, каждое сравнение в скобках; модуль — `abs()`; среднее и стандартное отклонение — методы Series `mean()` и `std()`: они пропускают пропуски, а массив `to_numpy()` с одним пропуском даёт пропуск во всех z-оценках; процентили — `values.quantile([0.01, 0.99])`, доли, а не проценты; `raise ValueError(...)` в конце функции, после трёх `if`.

Проверка в ноутбуке: стаж водителей области без кодов через flag_outliers; печатаются числа выбросов по правилам 'iqr', 'z' и 'percentile', все три вводятся в тест.

In [ ]:
def flag_outliers(values, method='iqr'):
    """
    Отмечает выбросы по правилу IQR, трёх сигм или процентилей.

    Три правила в одной функции, чтобы сравнивать их на одном столбце: в разделе
    2 ноутбука занятия они дали три разных числа. Пропуск не выброс: сравнение с
    пропуском и так даёт `False`, а маска через `~` и `between` объявила бы его
    выбросом. Правило трёх сигм на малой выборке слепо: среди 1, 2, 3, 4 и 100 у
    сотни z-оценка всего 1,79.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        method: Правило, строка: 'iqr' — за границами iqr_bounds с множителем
                1.5; 'z' — z-оценка (значение минус среднее, делённое на
                стандартное отклонение) по модулю больше 3; 'percentile' — ниже
                1-го или выше 99-го процентиля. По умолчанию 'iqr'.

    Возвращаемое значение:
        Объект Series из логических значений с тем же индексом: True у выбросов,
        False у остальных значений и у пропусков; значение ровно на границе — не
        выброс. Для другого method — ValueError. Series не меняется.
    """

    pass

In [ ]:
def flag_outliers_tests():
    values = pd.Series([1, 2, 3, 4, 100, None], index=[10, 11, 12, 13, 14, 15])
    before = values.copy()
    result = flag_outliers(values)
    assert isinstance(result, pd.Series) and result.dtype == bool, 'функция должна возвращать логический Series'
    assert result.index.tolist() == [10, 11, 12, 13, 14, 15], 'индекс как у values'
    assert result.tolist() == [False, False, False, False, True, False], "правило 'iqr' по умолчанию; пропуск — не выброс"
    assert flag_outliers(pd.Series([1, 2, 3, 4, 8])).tolist() == [False, False, False, False, True], 'множитель 1.5'
    assert not flag_outliers(pd.Series([1, 2, 3, 4, 7])).any(), 'значение на границе — не выброс'
    assert flag_outliers(values, 'z').tolist() == [False] * 6, 'на пяти значениях z-оценка сотни 1,79: выбросов нет'
    low = pd.Series([0] * 10 + [-100])
    assert flag_outliers(low, 'z').tolist() == [False] * 10 + [True], 'правило трёх сигм по модулю: выброс бывает и снизу'
    gap = pd.Series([0] * 10 + [-100, None])
    assert flag_outliers(gap, 'z').tolist() == [False] * 10 + [True, False], 'mean и std у Series пропускают пропуски: один пропуск не должен портить все z-оценки'
    assert flag_outliers(values, 'percentile').tolist() == [True, False, False, False, True, False], 'ниже 1-го и выше 99-го процентиля'
    try:
        flag_outliers(values, 'median')
    except ValueError:
        pass
    else:
        raise AssertionError('неизвестное правило должно давать ValueError')
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
flag_outliers_tests()

In [ ]:
# Проверка: стаж водителей области без кодов через flag_outliers; печатаются числа выбросов по правилам 'iqr', 'z' и 'percentile', все три вводятся в тест

people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель']

experience = drivers_check['experience_clean']
print(flag_outliers(experience, 'iqr').sum(), flag_outliers(experience, 'z').sum(), flag_outliers(experience, 'percentile').sum())

### Б3. Таблица выбросов по столбцам

Раздел «Аномалии» этапа 2 начинается с такой таблицы по всем числовым столбцам. Она же показывает, где правила ломаются: у счётчика с кучей нулей правило IQR объявляет выбросом всё, что не ноль. У столбца из одинаковых значений выбросов нет ни по одному правилу: квартили совпадают, а стандартное отклонение ноль, и z-оценка становится пропуском.

Реализуйте функцию `outlier_table(df, columns)`, которая считает выбросы по трём правилам для каждого столбца из списка. Аргументы: `df` — таблица; `columns` — список имён числовых столбцов. Возвращаемое значение: объект DataFrame со строками по columns в том же порядке и столбцами iqr, z, percentile: число выбросов столбца по каждому правилу, целые числа, через flag_outliers. Таблица не меняется.

Подсказка: раздел 3 ноутбука занятия; `flag_outliers(df[column], method).sum()` — число выбросов; таблицу удобно начать с `pd.DataFrame(index=columns)` и добавлять по столбцу на правило: список чисел по столбцам из `columns`. `df.columns` вместо `columns` отправит в `quantile` и текстовые столбцы.

Проверка в ноутбуке: четыре счётчика таблицы ДТП области через outlier_table; печатается таблица, в тест вводятся число выбросов vehicles_count по правилу 'z' и число выбросов injured_count по правилу 'percentile'.

In [ ]:
def outlier_table(df, columns):
    """
    Считает выбросы по трём правилам для каждого столбца из списка.

    Раздел «Аномалии» этапа 2 начинается с такой таблицы по всем числовым
    столбцам. Она же показывает, где правила ломаются: у счётчика с кучей нулей
    правило IQR объявляет выбросом всё, что не ноль. У столбца из одинаковых
    значений выбросов нет ни по одному правилу: квартили совпадают, а
    стандартное отклонение ноль, и z-оценка становится пропуском.

    Аргументы:
        df: Таблица.
        columns: Список имён числовых столбцов.

    Возвращаемое значение:
        Объект DataFrame со строками по columns в том же порядке и столбцами
        iqr, z, percentile: число выбросов столбца по каждому правилу, целые
        числа, через flag_outliers. Таблица не меняется.
    """

    pass

In [ ]:
def outlier_table_tests():
    df = pd.DataFrame({'a': [1, 2, 3, 4, 100, None], 'b': [5, 5, 5, 5, 5, 5], 'c': [1, 2, 3, 4, 5, 60], 'name': list('uvwxyz')})
    before = df.copy()
    table = outlier_table(df, ['a', 'b'])
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.index.tolist() == ['a', 'b'], 'строки — столбцы из списка columns'
    assert table.columns.tolist() == ['iqr', 'z', 'percentile'], 'столбцы — три правила'
    assert table.loc['a'].tolist() == [1, 0, 2], 'число выбросов по каждому правилу, не доля'
    assert table.loc['b'].tolist() == [0, 0, 0], 'у одинаковых значений выбросов нет'
    assert outlier_table(df, ['c', 'a']).index.tolist() == ['c', 'a'], 'строки в порядке columns'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
outlier_table_tests()

In [ ]:
# Проверка: четыре счётчика таблицы ДТП области через outlier_table; печатается таблица, в тест вводятся число выбросов vehicles_count по правилу 'z' и число выбросов injured_count по правилу 'percentile'

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

print(outlier_table(df_check, ['participants_count', 'vehicles_count', 'dead_count', 'injured_count']))

### Б4. Срез по процентилям

Срез, или винзоризация, — одно из пяти решений раздела 4 ноутбука занятия: крайние значения не удаляются, а прижимаются к краю. Среднее меняется, медиана нет, число строк то же. Края задаются процентилями: функция сама находит значения процентилей и передаёт их в `clip`, ведь доли 0.01 и 0.99 — не границы, а номера процентилей.

Реализуйте функцию `cap_values(values, low=0.01, high=0.99)`, которая прижимает крайние значения к процентилям. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `low` — нижний процентиль, доля от 0 до 1; по умолчанию 0.01; `high` — верхний процентиль, доля от 0 до 1; по умолчанию 0.99. Возвращаемое значение: объект Series с тем же индексом: значения ниже low-го процентиля подняты до него, выше high-го опущены до него, остальные как были; пропуски остаются пропусками. Series не меняется.

Подсказка: раздел 4 ноутбука занятия: `lower, upper = values.quantile([low, high])`, затем `values.clip(lower=lower, upper=upper)`.

Проверка в ноутбуке: возраст машин водителей области через cap_values; печатаются среднее до среза и после среза до тысячных, оба вводятся в тест.

In [ ]:
def cap_values(values, low=0.01, high=0.99):
    """
    Прижимает крайние значения к процентилям.

    Срез, или винзоризация, — одно из пяти решений раздела 4 ноутбука занятия:
    крайние значения не удаляются, а прижимаются к краю. Среднее меняется,
    медиана нет, число строк то же. Края задаются процентилями: функция сама
    находит значения процентилей и передаёт их в `clip`, ведь доли 0.01 и 0.99 —
    не границы, а номера процентилей.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        low: Нижний процентиль, доля от 0 до 1; по умолчанию 0.01.
        high: Верхний процентиль, доля от 0 до 1; по умолчанию 0.99.

    Возвращаемое значение:
        Объект Series с тем же индексом: значения ниже low-го процентиля подняты
        до него, выше high-го опущены до него, остальные как были; пропуски
        остаются пропусками. Series не меняется.
    """

    pass

In [ ]:
def cap_values_tests():
    values = pd.Series([1, 2, 3, 4, 100, None], index=list('abcdef'))
    before = values.copy()
    result = cap_values(values)
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.index.tolist() == list('abcdef'), 'индекс сохранён, строки не удаляются'
    assert result.round(2).tolist()[:5] == [1.04, 2.0, 3.0, 4.0, 96.16], 'края прижаты к 1-му и 99-му процентилям'
    assert pd.isna(result['f']), 'пропуск остаётся пропуском'
    assert cap_values(values, low=0.2, high=0.8).round(2).tolist()[:5] == [1.8, 2.0, 3.0, 4.0, 23.2], 'процентили low и high'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
cap_values_tests()

In [ ]:
# Проверка: возраст машин водителей области через cap_values; печатаются среднее до среза и после среза до тысячных, оба вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check[['datetime']], left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']

capped = cap_values(drivers_check['vehicle_age'])
print(round(drivers_check['vehicle_age'].mean(), 3), round(capped.mean(), 3))

### Б5. Невозможное в пропуск

Заменить пропуском — одно из пяти решений раздела 4 ноутбука занятия: невозможное значение, которое не восстановить, не удаляется вместе со строкой, а становится пропуском, и строка остаётся для остальных столбцов. Так поступают с годом выпуска 1, с нулевыми координатами, с датой из будущего. Переставленные широту и долготу пропуском не заменяют, а исправляют — меняют обратно, как в разделе 4; функция такие пары от ошибок не отличает. Границы включаются: если допустимы годы от 1900-го, то 1900 год допустим.

Реализуйте функцию `nan_outside(values, low, high)`, которая заменяет пропуском значения вне допустимых границ. Аргументы: `values` — столбец чисел, Series; `low` — наименьшее допустимое значение, число; само допустимо; `high` — наибольшее допустимое значение, число; само допустимо. Возвращаемое значение: объект Series с тем же индексом и той же длины: значения вне границ low и high заменены пропуском, остальные как были; пропуски остаются пропусками. Series не меняется.

Подсказка: раздел 4 ноутбука занятия, решение «заменить пропуском»: `values.between(low, high)` — маска допустимых значений, обе границы включаются, как в проверках раздела 6; `values.where(маска)` оставляет значения там, где маска истинна, остальные заменяет пропуском.

Проверка в ноутбуке: широта ДТП Липецкой области с границами 51.8 и 53.7 через nan_outside; печатаются число новых пропусков и наименьшая оставшаяся широта до тысячных, оба вводятся в тест.

In [ ]:
def nan_outside(values, low, high):
    """
    Заменяет пропуском значения вне допустимых границ.

    Заменить пропуском — одно из пяти решений раздела 4 ноутбука занятия:
    невозможное значение, которое не восстановить, не удаляется вместе со
    строкой, а становится пропуском, и строка остаётся для остальных столбцов.
    Так поступают с годом выпуска 1, с нулевыми координатами, с датой из
    будущего. Переставленные широту и долготу пропуском не заменяют, а
    исправляют — меняют обратно, как в разделе 4; функция такие пары от ошибок
    не отличает. Границы включаются: если допустимы годы от 1900-го, то 1900 год
    допустим.

    Аргументы:
        values: Столбец чисел, Series.
        low: Наименьшее допустимое значение, число; само допустимо.
        high: Наибольшее допустимое значение, число; само допустимо.

    Возвращаемое значение:
        Объект Series с тем же индексом и той же длины: значения вне границ low
        и high заменены пропуском, остальные как были; пропуски остаются
        пропусками. Series не меняется.
    """

    pass

In [ ]:
def nan_outside_tests():
    values = pd.Series([1, 2000, 2015, 2030, 1900, None], index=[3, 4, 5, 6, 7, 8])
    before = values.copy()
    result = nan_outside(values, 1900, 2025)
    assert isinstance(result, pd.Series) and len(result) == 6, 'Series той же длины: строки не удаляются'
    assert result.index.tolist() == [3, 4, 5, 6, 7, 8], 'индекс сохранён'
    assert result.isna().tolist() == [True, False, False, True, False, True], 'значения вне границ — пропуск'
    assert result.dropna().tolist() == [2000, 2015, 1900], 'остальные значения как были, граница включается'
    assert nan_outside(values, 2000, 2015).dropna().tolist() == [2000, 2015], 'обе границы включаются'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
nan_outside_tests()

In [ ]:
# Проверка: широта ДТП Липецкой области с границами 51.8 и 53.7 через nan_outside; печатаются число новых пропусков и наименьшая оставшаяся широта до тысячных, оба вводятся в тест

lipetsk_check = pd.read_csv('../datasets/legal/dtp-lipetsk.csv', index_col='id', parse_dates=['datetime'])

lat = nan_outside(lipetsk_check['lat'], 51.8, 53.7)
print(lat.isna().sum() - lipetsk_check['lat'].isna().sum(), round(lat.min(), 3))

### Б6. Заполнение медианой группы

Заполнять пропуск медианой всей таблицы грубо, медианой группы честнее: стаж различается у женщин и мужчин, возраст машины — у грузовика и легковой машины. `transform('median')` возвращает Series длины таблицы — медиану группы на каждой строке, — и `fillna` принимает его как есть. Группа, где значений нет совсем, остаётся с пропуском: заполнять её нечем.

Реализуйте функцию `fill_group_median(df, column, group)`, которая заполняет пропуски медианой своей группы. Аргументы: `df` — таблица; `column` — имя числового столбца с пропусками, строка; `group` — имя столбца групп, строка. Возвращаемое значение: объект Series с тем же индексом, что у таблицы: значения column, где пропуски заполнены медианой column в своей группе; если в группе нет ни одного значения, пропуск остаётся. Возвращается Series, а не таблица; таблица не меняется.

Подсказка: раздел 5 ноутбука занятия: `df.groupby(group)[column].transform('median')`, затем `df[column].fillna(...)`. `groupby(...).median()` без `transform` даёт Series с группами в индексе, и `fillna` не находит в нём меток строк.

Проверка в ноутбуке: возраст машин водителей области, заполненный медианой класса машины, через fill_group_median; печатаются число оставшихся пропусков и среднее после заполнения до тысячных, оба вводятся в тест.

In [ ]:
def fill_group_median(df, column, group):
    """
    Заполняет пропуски медианой своей группы.

    Заполнять пропуск медианой всей таблицы грубо, медианой группы честнее: стаж
    различается у женщин и мужчин, возраст машины — у грузовика и легковой
    машины. `transform('median')` возвращает Series длины таблицы — медиану
    группы на каждой строке, — и `fillna` принимает его как есть. Группа, где
    значений нет совсем, остаётся с пропуском: заполнять её нечем.

    Аргументы:
        df: Таблица.
        column: Имя числового столбца с пропусками, строка.
        group: Имя столбца групп, строка.

    Возвращаемое значение:
        Объект Series с тем же индексом, что у таблицы: значения column, где
        пропуски заполнены медианой column в своей группе; если в группе нет ни
        одного значения, пропуск остаётся. Возвращается Series, а не таблица;
        таблица не меняется.
    """

    pass

In [ ]:
def fill_group_median_tests():
    df = pd.DataFrame({'g': ['a', 'a', 'a', 'b', 'b', 'c'], 'v': [1, None, 3, 10, None, None]})
    before = df.copy()
    result = fill_group_median(df, 'v', 'g')
    assert isinstance(result, pd.Series), 'функция должна возвращать Series, а не таблицу'
    assert result.index.equals(df.index), 'индекс как у таблицы'
    assert result.tolist()[:5] == [1, 2, 3, 10, 10], 'пропуск заполнен медианой своей группы'
    assert pd.isna(result.iloc[5]), 'в группе без значений пропуск остаётся'
    other = pd.DataFrame({'g': ['x', 'y', 'x', 'y', 'x', 'x'], 'v': [1, 5, 2, None, None, 9]}, index=[7, 3, 5, 1, 9, 2])
    assert fill_group_median(other, 'v', 'g').tolist() == [1, 5, 2, 5, 2, 9], 'медиана группы, а не среднее и не медиана всей таблицы'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
fill_group_median_tests()

In [ ]:
# Проверка: возраст машин водителей области, заполненный медианой класса машины, через fill_group_median; печатаются число оставшихся пропусков и среднее после заполнения до тысячных, оба вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check[['datetime']], left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']

filled = fill_group_median(drivers_check, 'vehicle_age', 'vehicle_category')
print(filled.isna().sum(), round(filled.mean(), 3))

### Б7. Доля пропусков по группам

Пропуск со смыслом виден только по группам: стаж пуст у всех пешеходов и лишь у малой части водителей — это две разные истории, и одна функция их различает. Из неё строится раздел «Пропуски» этапа 2 по группам; самые дырявые группы — первыми.

Реализуйте функцию `missing_by_group(df, column, group)`, которая считает долю пропусков столбца в каждой группе. Аргументы: `df` — таблица; `column` — имя столбца, строка; `group` — имя столбца групп, строка. Возвращаемое значение: объект Series долей пропусков column в каждой группе group, от 0 до 1, отсортированный по убыванию доли; индекс — значения group. Таблица не меняется.

Подсказка: раздел 5 ноутбука занятия: `df[column].isna()` — маска пропусков, её среднее по группам — доля: `df[column].isna().groupby(df[group]).mean()`; затем `sort_values(ascending=False)`.

Проверка в ноутбуке: год выпуска по ролям в таблице участников области через missing_by_group; печатаются доля пропусков у водителей до тысячных и число ролей, где год выпуска пропущен у всех, оба вводятся в тест.

In [ ]:
def missing_by_group(df, column, group):
    """
    Считает долю пропусков столбца в каждой группе.

    Пропуск со смыслом виден только по группам: стаж пуст у всех пешеходов и
    лишь у малой части водителей — это две разные истории, и одна функция их
    различает. Из неё строится раздел «Пропуски» этапа 2 по группам; самые
    дырявые группы — первыми.

    Аргументы:
        df: Таблица.
        column: Имя столбца, строка.
        group: Имя столбца групп, строка.

    Возвращаемое значение:
        Объект Series долей пропусков column в каждой группе group, от 0 до 1,
        отсортированный по убыванию доли; индекс — значения group. Таблица не
        меняется.
    """

    pass

In [ ]:
def missing_by_group_tests():
    df = pd.DataFrame({'g': ['a', 'a', 'a', 'b', 'b', 'c'], 'v': [1, None, 3, 10, None, None]})
    before = df.copy()
    result = missing_by_group(df, 'v', 'g')
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.round(3).to_dict() == {'c': 1.0, 'b': 0.5, 'a': 0.333}, 'доля пропусков в каждой группе, а не их число'
    assert result.index.tolist() == ['c', 'b', 'a'], 'по убыванию доли'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
missing_by_group_tests()

In [ ]:
# Проверка: год выпуска по ролям в таблице участников области через missing_by_group; печатаются доля пропусков у водителей до тысячных и число ролей, где год выпуска пропущен у всех, оба вводятся в тест

people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')

shares = missing_by_group(people_check, 'vehicle_year', 'role')
print(round(shares['Водитель'], 3))
print((shares == 1).sum())

### Б8. Логические проверки таблицы ДТП

Проверки, которые ничего не нашли, в отчёт тоже идут: читатель должен знать, что правило проверено. Четыре правила фиксированы, чтобы функция годилась для любой из трёх таблиц ДТП курса и для контрольной: пострадавших, то есть погибших и раненых вместе, не больше участников; машин не больше участников; машины есть; участники есть. Равенство — не нарушение: ДТП с одним участником и одним раненым возможно.

Реализуйте функцию `logic_checks(df)`, которая считает нарушения четырёх логических правил таблицы ДТП. Аргументы: `df` — таблица ДТП со столбцами participants_count, vehicles_count, dead_count, injured_count. Возвращаемое значение: объект Series из четырёх целых чисел с именами «пострадавших больше участников», «машин больше участников», «машин нет», «участников нет» в этом порядке: число строк, нарушающих каждое правило. Таблица не меняется.

Подсказка: раздел 6 ноутбука занятия: каждое правило — маска, её сумма — число нарушений; `pd.Series({'имя': число, ...})` собирает числа с именами. Пострадавшие — сумма `dead_count + injured_count`, а не каждый столбец по отдельности.

Проверка в ноутбуке: таблица ДТП Липецкой области через logic_checks; печатается Series, в тест вводятся число ДТП, где машин больше участников, и число правил без нарушений.

In [ ]:
def logic_checks(df):
    """
    Считает нарушения четырёх логических правил таблицы ДТП.

    Проверки, которые ничего не нашли, в отчёт тоже идут: читатель должен знать,
    что правило проверено. Четыре правила фиксированы, чтобы функция годилась
    для любой из трёх таблиц ДТП курса и для контрольной: пострадавших, то есть
    погибших и раненых вместе, не больше участников; машин не больше участников;
    машины есть; участники есть. Равенство — не нарушение: ДТП с одним
    участником и одним раненым возможно.

    Аргументы:
        df: Таблица ДТП со столбцами participants_count, vehicles_count,
            dead_count, injured_count.

    Возвращаемое значение:
        Объект Series из четырёх целых чисел с именами «пострадавших больше
        участников», «машин больше участников», «машин нет», «участников нет» в
        этом порядке: число строк, нарушающих каждое правило. Таблица не
        меняется.
    """

    pass

In [ ]:
def logic_checks_tests():
    df = pd.DataFrame({'participants_count': [2, 1, 0, 2, 3], 'vehicles_count': [1, 2, 0, 2, 1],
                       'dead_count': [1, 0, 0, 0, 1], 'injured_count': [2, 1, 0, 2, 2]})
    before = df.copy()
    result = logic_checks(df)
    names = ['пострадавших больше участников', 'машин больше участников', 'машин нет', 'участников нет']
    assert isinstance(result, pd.Series), 'функция должна возвращать Series с именами правил'
    assert result.index.tolist() == names, 'имена строк как в условии, в том же порядке'
    assert result.tolist() == [1, 1, 1, 1], 'по одной строке с нарушением каждого правила; равенство — не нарушение'
    clean = pd.DataFrame({'participants_count': [1, 4], 'vehicles_count': [1, 2], 'dead_count': [0, 1], 'injured_count': [1, 3]})
    assert logic_checks(clean).tolist() == [0, 0, 0, 0], 'на чистой таблице нули'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
logic_checks_tests()

In [ ]:
# Проверка: таблица ДТП Липецкой области через logic_checks; печатается Series, в тест вводятся число ДТП, где машин больше участников, и число правил без нарушений

lipetsk_check = pd.read_csv('../datasets/legal/dtp-lipetsk.csv', index_col='id', parse_dates=['datetime'])

print(logic_checks(lipetsk_check))

## Часть В. Задачи повышенной сложности

### В1. Таблица «до и после»

Цена решения видна только рядом: среднее меняется, медиана нет. Удаление и пропуск дают одну сводку, но разную таблицу: пропуск сохраняет строку для остальных столбцов. Функция собирает строки таблицы раздела 4 ноутбука занятия из функций Б1 и Б4.

Реализуйте функцию `before_after(values)`, которая сравнивает среднее, медиану и число значений после разных решений о выбросах. Аргументы: `values` — столбец чисел, Series; пропуски допускаются. Возвращаемое значение: объект DataFrame с четырьмя строками «как есть», «срез по 99-му процентилю», «удалить выше границы IQR», «пропуск выше границы IQR» в этом порядке и столбцами «среднее», «медиана», «n»: среднее, округлённое до сотых, медиана и число непустых значений. Срез — через cap_values с процентилями по умолчанию; граница — верхняя граница iqr_bounds, значение ровно на ней не удаляется. Series не меняется.

Подсказка: раздел 4 ноутбука занятия: четыре варианта Series — `values`, `cap_values(values)`, `values[values <= upper]` и `values.where(values <= upper)`; у каждого `round(v.mean(), 2)`, `v.median()` и `v.count()`; таблица — `pd.DataFrame({'среднее': [...], 'медиана': [...], 'n': [...]}, index=[...])`. `len` считает и пропуски, `count` — нет.

Проверка в ноутбуке: возраст машин водителей области через before_after; печатается таблица, в тест вводятся среднее и n в строке «удалить выше границы IQR».

In [ ]:
def before_after(values):
    """
    Сравнивает среднее, медиану и число значений после разных решений о
    выбросах.

    Цена решения видна только рядом: среднее меняется, медиана нет. Удаление и
    пропуск дают одну сводку, но разную таблицу: пропуск сохраняет строку для
    остальных столбцов. Функция собирает строки таблицы раздела 4 ноутбука
    занятия из функций Б1 и Б4.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.

    Возвращаемое значение:
        Объект DataFrame с четырьмя строками «как есть», «срез по 99-му
        процентилю», «удалить выше границы IQR», «пропуск выше границы IQR» в
        этом порядке и столбцами «среднее», «медиана», «n»: среднее, округлённое
        до сотых, медиана и число непустых значений. Срез — через cap_values с
        процентилями по умолчанию; граница — верхняя граница iqr_bounds,
        значение ровно на ней не удаляется. Series не меняется.
    """

    pass

In [ ]:
def before_after_tests():
    values = pd.Series([1, 2, 3, 4, 100, None])
    before = values.copy()
    table = before_after(values)
    rows = ['как есть', 'срез по 99-му процентилю', 'удалить выше границы IQR', 'пропуск выше границы IQR']
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.index.tolist() == rows, 'строки как в условии, в том же порядке'
    assert table.columns.tolist() == ['среднее', 'медиана', 'n'], 'столбцы как в условии'
    assert table.loc['как есть'].tolist() == [22.0, 3.0, 5], 'n — число непустых значений'
    assert table.loc['срез по 99-му процентилю'].tolist() == [21.24, 3.0, 5], 'срез по 1-му и 99-му процентилям через cap_values'
    assert table.loc['удалить выше границы IQR'].tolist() == [2.5, 2.5, 4], 'удалены значения выше верхней границы IQR'
    assert table.loc['пропуск выше границы IQR'].tolist() == [2.5, 2.5, 4], 'значения выше границы стали пропуском'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
before_after_tests()

In [ ]:
# Проверка: возраст машин водителей области через before_after; печатается таблица, в тест вводятся среднее и n в строке «удалить выше границы IQR»

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check[['datetime']], left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']

print(before_after(drivers_check['vehicle_age']))

### В2. Отчёт о повторах по группам

Прежде чем удалять повторы, надо увидеть, кто они. В таблице участников это пассажиры одной машины с одинаковыми полом и здоровьем и пустым стажем, и удалять их нельзя: это разные люди. Отчёт ставит рядом размер группы и число её строк, входящих в группы одинаковых, — так видно, у кого повторы и объяснимы ли они.

Реализуйте функцию `duplicate_report(df, group)`, которая считает строки в группах полных повторов для каждой группы. Аргументы: `df` — таблица; `group` — имя столбца групп, строка: роль, вид ДТП. Возвращаемое значение: объект DataFrame со строками по значениям group и столбцами «строк» — число строк группы и «в повторах» — число строк группы, входящих в группы полных повторов всей таблицы, как их помечает duplicated(keep=False); у групп без повторов ноль; строки отсортированы по убыванию «в повторах». Таблица не меняется.

Подсказка: раздел 6 ноутбука занятия: `repeated = df.duplicated(keep=False)` — маска по всей таблице; `df.groupby(group).size()` — строк в группе; `repeated.groupby(df[group]).sum()` — строк в повторах; оба Series — в один `pd.DataFrame({...})`, затем `sort_values('в повторах', ascending=False)`.

Проверка в ноутбуке: таблица участников области по ролям через duplicate_report; печатаются три первые строки, в тест вводятся числа пассажиров и пешеходов в повторах.

In [ ]:
def duplicate_report(df, group):
    """
    Считает строки в группах полных повторов для каждой группы.

    Прежде чем удалять повторы, надо увидеть, кто они. В таблице участников это
    пассажиры одной машины с одинаковыми полом и здоровьем и пустым стажем, и
    удалять их нельзя: это разные люди. Отчёт ставит рядом размер группы и число
    её строк, входящих в группы одинаковых, — так видно, у кого повторы и
    объяснимы ли они.

    Аргументы:
        df: Таблица.
        group: Имя столбца групп, строка: роль, вид ДТП.

    Возвращаемое значение:
        Объект DataFrame со строками по значениям group и столбцами «строк» —
        число строк группы и «в повторах» — число строк группы, входящих в
        группы полных повторов всей таблицы, как их помечает
        duplicated(keep=False); у групп без повторов ноль; строки отсортированы
        по убыванию «в повторах». Таблица не меняется.
    """

    pass

In [ ]:
def duplicate_report_tests():
    df = pd.DataFrame({'g': ['a', 'a', 'a', 'a', 'b', 'b'], 'x': [1, 1, 2, 2, 1, 2]})
    before = df.copy()
    report = duplicate_report(df, 'g')
    assert isinstance(report, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert report.columns.tolist() == ['строк', 'в повторах'], 'столбцы как в условии'
    assert report.index.tolist() == ['a', 'b'], 'строки — значения group, по убыванию «в повторах»'
    assert report.loc['a'].tolist() == [4, 4], 'в повторах все строки группы, не только вторые вхождения'
    assert report.loc['b'].tolist() == [2, 0], 'у группы без повторов ноль'
    other = pd.DataFrame({'g': ['p', 'q', 'q', 'r', 'p', 'r', 'q', 'q', 'r', 'q'], 'x': [5, 1, 2, 7, 5, 7, 3, 4, 7, 6]})
    assert duplicate_report(other, 'g').index.tolist() == ['r', 'p', 'q'], 'по убыванию «в повторах», а не «строк»'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
duplicate_report_tests()

In [ ]:
# Проверка: таблица участников области по ролям через duplicate_report; печатаются три первые строки, в тест вводятся числа пассажиров и пешеходов в повторах

people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')

print(duplicate_report(people_check, 'role').head(3))

### В3. Очистка таблицы участников с журналом

Решения разделов 4 и 5 ноутбука занятия собираются в одну функцию с журналом: так раздел «Аномалии» выглядит в коде, а не в словах. Коды стажа становятся пропуском, невозможный год выпуска — тоже, а пропуск стажа у водителя сохраняется индикатором. Журнал печатается рядом с очищенной таблицей и переходит в отчёт.

Реализуйте функцию `clean_participants(people, year_max)`, которая очищает таблицу участников и возвращает журнал решений. Аргументы: `people` — таблица участников со столбцами role, experience, vehicle_year; `year_max` — последний допустимый год выпуска, целое число. Возвращаемое значение: кортеж (clean, log). clean — копия таблицы той же длины: новый столбец experience_clean, где коды стажа 95, 96 и 99 заменены пропуском; vehicle_year, где годы вне 1900–year_max заменены пропуском, обе границы допустимы; логический столбец experience_missing — пропуск experience_clean у водителя, у остальных ролей False. log — Series с именами «коды стажа в пропуск», «год выпуска вне диапазона в пропуск», «водителей без стажа» и числами затронутых строк. Входная таблица не меняется.

Подсказка: `clean = people.copy()`; `isin([95, 96, 99])` — маска кодов, `replace([95, 96, 99], np.nan)` с присваиванием в новый столбец; годы вне диапазона — маска из двух сравнений через `|`, пропуск — `clean.loc[маска, 'vehicle_year'] = np.nan`; сравнение с пропуском даёт `False`, поэтому пустые годы в маску не попадут; журнал — `pd.Series({'имя': число, ...})`; `return clean, log`.

Проверка в ноутбуке: таблица участников области с year_max=2025 через clean_participants; печатается журнал, в тест вводятся число кодов стажа и число водителей без стажа.

In [ ]:
def clean_participants(people, year_max):
    """
    Очищает таблицу участников и возвращает журнал решений.

    Решения разделов 4 и 5 ноутбука занятия собираются в одну функцию с
    журналом: так раздел «Аномалии» выглядит в коде, а не в словах. Коды стажа
    становятся пропуском, невозможный год выпуска — тоже, а пропуск стажа у
    водителя сохраняется индикатором. Журнал печатается рядом с очищенной
    таблицей и переходит в отчёт.

    Аргументы:
        people: Таблица участников со столбцами role, experience, vehicle_year.
        year_max: Последний допустимый год выпуска, целое число.

    Возвращаемое значение:
        Кортеж (clean, log). clean — копия таблицы той же длины: новый столбец
        experience_clean, где коды стажа 95, 96 и 99 заменены пропуском;
        vehicle_year, где годы вне 1900–year_max заменены пропуском, обе границы
        допустимы; логический столбец experience_missing — пропуск
        experience_clean у водителя, у остальных ролей False. log — Series с
        именами «коды стажа в пропуск», «год выпуска вне диапазона в пропуск»,
        «водителей без стажа» и числами затронутых строк. Входная таблица не
        меняется.
    """

    pass

In [ ]:
def clean_participants_tests():
    people = pd.DataFrame({'role': ['Водитель', 'Водитель', 'Водитель', 'Пассажир', 'Водитель', 'Пешеход'],
                           'experience': [5, 95, None, None, 30, None],
                           'vehicle_year': [2010, 1, 2030, 2025, 1900, None]})
    before = people.copy()
    result = clean_participants(people, 2025)
    assert people.equals(before), 'исходная таблица изменилась: работайте с копией people.copy()'
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (clean, log)'
    clean, log = result
    names = ['коды стажа в пропуск', 'год выпуска вне диапазона в пропуск', 'водителей без стажа']
    assert isinstance(log, pd.Series) and log.index.tolist() == names, 'журнал — Series с именами как в условии'
    assert log.tolist() == [1, 2, 2], 'журнал: коды один, годы вне диапазона два, водителей без стажа два'
    assert len(clean) == 6 and clean.index.equals(people.index), 'clean той же длины: строки не удаляются'
    assert clean['experience_clean'].isna().tolist() == [False, True, True, True, False, True], 'код стажа стал пропуском'
    assert clean['experience_clean'].dropna().tolist() == [5, 30]
    assert clean['vehicle_year'].isna().tolist() == [False, True, True, False, False, True], 'годы вне 1900–year_max стали пропуском, границы допустимы'
    assert clean['experience_missing'].tolist() == [False, True, True, False, False, False], 'пропуск стажа отмечен только у водителей'
    assert clean_participants(people, 2030)[1].tolist() == [1, 1, 2], 'последний допустимый год — year_max'

    print('Все тесты прошли успешно!')

In [ ]:
clean_participants_tests()

In [ ]:
# Проверка: таблица участников области с year_max=2025 через clean_participants; печатается журнал, в тест вводятся число кодов стажа и число водителей без стажа

people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')

clean, log = clean_participants(people_check, 2025)
print(log)

## Необязательная задача. Раздел «Аномалии» этапа 2 для своего датасета

Этап 2 проекта, `project/02-eda.md`, сдаётся на семинаре 11, и его раздел «Выявление аномалий и проблем» требует трёх вещей: выбросы по числовым столбцам, логические проверки, дубликаты. Найти мало: по каждой находке нужно решение и его причина. Раздел для своего датасета сейчас — готовая часть отчёта этапа 2: он переходит в отчёт как есть и сдаётся вместе с ним на семинаре 11.

Состав — функциями частей Б и В: таблица выбросов по числовым столбцам — `outlier_table`; логические проверки по смыслу своих столбцов — две-три маски по образцу Б8, а если это таблица ДТП — `logic_checks`; повторы по группам — `duplicate_report`; по каждому найденному — решение: оставить и пометить, исправить, удалить, срезать через `cap_values`, заменить пропуском через `nan_outside`; цену решения показывает `before_after`.

Журнал решений — таблица Markdown со столбцами «Проверка», «Правило», «Найдено», «Решение», как в разделе 7 ноутбука занятия: в «Решении» сначала что сделано, а после двоеточия — почему; проверки без находок тоже записываются. Заготовка — последняя ячейка ноутбука. Памятка из пяти вопросов к выбросу:
1. Возможно ли значение?
2. Повторяется ли?
3. Объяснимо ли?
4. Что меняет в сводке?
5. Что записано в журнал?

Если датасета ещё нет или в нём нет подходящих столбцов, возьмите таблицы Брянской области `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv`: в упражнениях они не использовались. Проверки, уже сделанные в разделе 7 ноутбука занятия, не повторяйте, а замените другими: например, тяжесть ДТП `severity` против числа погибших и раненых, число нарушений `violations_count` против текста `violation`, счётчик недостатков дороги `road_defects` против `road_defect`, число участников `participants_count` против числа строк ДТП в таблице участников.

В ответ в тест Moodle вставьте код и журнал решений; вложений у этого вопроса нет. Балл ставит преподаватель по памятке из пяти вопросов; автоматической проверки нет.

In [ ]:
path = '../datasets/legal/dtp-bryansk.csv'   # путь к своему файлу; пока таблица ДТП Брянской области
data = pd.read_csv(path)

print(data.shape)
print(data.dtypes)

In [ ]:
# таблица выбросов по числовым столбцам

In [ ]:
# логические проверки по смыслу столбцов

In [ ]:
# повторы по группам

In [ ]:
# решения и их цена

Журнал решений

| Проверка | Правило | Найдено | Решение |
|---|---|---|---|
| | | | |